<a href="https://colab.research.google.com/github/minbj1226/computer_vision-study/blob/main/Computer_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 이미지 데이터는 어떻게 표현되는가?
- 배치 크기(m), 채널 수(c), 높이(h), 너비(w)로 표현이 된다.
- PyTorch는 (m, c, h, w) 형식을 주로 사용한다.

### 픽셀과 해상도는 무엇인가?
- 픽셀은 이미지를 이루는 가장 작은 단위로, 밝기 또는 색상 강도를 나타내며 0~255 범위를 가진다.
- 해상도는 이미지가 몇 개의 픽셀로 구성되어 있는지를 의미

### RGB 이미지와 Grayscale 이미지의 차이는 무엇인가?
- RGB 이미지는 channel이 Red, Green, Blue 3개 채널로 구성
- Grayscale 이미지는 밝기 정보만 가지므로 1개의 채널이다.

### 이미지의 channel은 무엇을 의미하는가?
- 채널은 이미지나 feature map이 가지는 정보의 종류를 의미

### Convolution 연산이란 무엇인가?
- 가중치를 담당하는 Kernel을 입력 위에 슬라이딩하여 커널의 가중치와 입력의 한 영역을 곱하고 모두 더해 새로운 특징값을 만드는 과정

### Kernel(Filter)는 어떤 역할을 하는가?
- Kernel은 학습되는 가중치들의 집합으로, 입력 이미지에서 특정 패턴을 추출하는 역할을 한다.

### Stride는 무엇이며 출력 크기에 어떤 영향을 주는가?
- Kernel이 입력 위를 한 번에 이동하는 칸수로, Stride의 값이 크면 feature map의 높이와 너비가 작아진다.

### Padding은 왜 사용하는가?
- 입력 가장자리에 값을 추가하여, convolution 후 출력 크기 감소를 완화하고 경계 정보 손실을 줄이기 위해 사용한다.

### Pooling은 무엇이며 왜 사용하는가?
- 입력의 일정 영역에서 대표값을 선택하여 공간 크기를 줄이는 연산이다.
- 연산량을 줄이고, 중요한 특징을 요약할 수 있다.

### Feature Map이란 무엇인가?
- convolution 연산의 결과로 얻어지는 출력으로, 입력 이미지에서 추출된 특정 특징들의 활성화 값을 나타낸다.

### Receptive Field란 무엇인가?
- CNN의 특정 뉴런이 입력 이미지에서 영향을 받는 영역의 크기를 의미
- 층이 깊어질수록 receptive field는 점점 넓어진다.

### CNN에서 layer가 깊어질수록 어떤 특징을 보게 되는가?
- 초기 층에서는 edge, corner 같은 저수준 특징을 학습
- 중간 층에서는 패턴, 질감 같은 중간 수준 특징을 학습
- 깊은 층에서는 객체의 형태나 클래스에 가까운 고수준 의미적 특징을 학습

# Convolutional Neural Networks
## 1. Zero-Padding
- 이미지의 높이와 너비 가장자리에 0을 추가하여, 컨볼루션 후 출력 크기 감소와 경계 정보 손실을 줄이는 기법

## 2. conv_single_step
- 입력의 한 영역과 필터 가중치를 곱해 합을 구한 뒤, 편향을 더해 하나의 출력값을 만드는 연산

## 3. conv_forward
- 필터를 입력 전체에 슬라이딩하며 각 위치에서 입력의 한 영역과 연산해 출력 feature map을 만드는 과정

## 4. pool_forward
- 입력의 일정 영역에서 최댓값 또는 평균값을 선택하여 공간 크기를 줄이고 주요 특징을 요약하는 과정

## 5. conv_backward
- 컨볼루션 층의 역전파 과정으로, 출력 gradient를 이용해 입력, 가중치, 편향에 대한 gradient를 계산하는 과정

## 6. create_mask_from_window
- max pooling 역전파에서 윈도우 내 최댓값의 위치만 True로 표시하는 마스크를 만드는 과정

## 7. distribute_value
- average pooling 역전파에서 출력 gradient 값을 윈도우의 모든 위치에 균등하게 나누어 분배하는 과정

## 8. pool_backward
- pooling 층의 역전파 과정으로, max pooling은 최댓값 위치만 gradient를 전달하고 average pooling은 전체에 균등 분배하는 과정

In [1]:
# zero_padding

def zero_pad(X, pad):
    X_pad = np.pad(X, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode='constant', constant_values=0)

    return X_pad

In [2]:
# conv_single(컨볼루션 계산 과정)

import numpy as np

a_slice_prev = np.random.randn(4, 4, 3)
W = np.random.randn(4, 4, 3)
b = np.random.randn(1, 1, 1)

def conv_single_step(a_slice_prev, W, b):
    s = a_slice_prev * W
    Z = np.sum(s)
    Z = Z + float(b)

    return Z

In [3]:
# conv_forward

np.random.seed(1)
A_prev = np.random.randn(2, 5, 7, 4)
W = np.random.randn(3, 3, 4, 8)
b = np.random.randn(1, 1, 1, 8)
hparameters = {"pad" : 1,
               "stride": 2}


def conv_forward(A_prev, W, b, hparameters):
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape

    (f, f, n_C_prev, n_C) = W.shape

    stride = hparameters['stride']
    pad = hparameters['pad']

    n_H = int((n_H_prev+2*pad-f)/stride) + 1
    n_W = int((n_W_prev+2*pad-f)/stride) + 1

    Z = np.zeros((m, n_H, n_W, n_C))

    A_prev_pad = zero_pad(A_prev, pad)

    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        for h in range(n_H):
            vert_start = h * stride
            vert_end = vert_start + f

            for w in range(n_W):
                horiz_start = w * stride
                horiz_end = horiz_start + f

                for c in range(n_C):

                    a_slice_prev = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]

                    weights = W[:, :, :, c]
                    biases = b[:, :, :, c]
                    Z[i, h, w, c] = conv_single_step(a_slice_prev, weights, biases)

    cache = (A_prev, W, b, hparameters)

    return Z, cache

In [4]:
# pooling forward
def pool_forward(A_prev, hparameters, mode = "max"):
  (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape

  f = hparameters["f"]
  stride = hparameters["stride"]

  n_H = int(1 + (n_H_prev - f) / stride)
  n_W = int(1 + (n_W_prev - f) / stride)
  n_C = n_C_prev

  A = np.zeros((m, n_H, n_W, n_C))

  for i in range(m):
    for h in range(n_H):
      vert_start = h * stride
      vert_end = vert_start + f

      for w in range(n_W):
        horiz_start = w * stride
        horiz_end = horiz_start + f

        for c in range(n_C):
          a_prev_slice = A_prev[i, vert_start:vert_end, horiz_start:horiz_end, c]

          if mode == "max":
            A[i, h, w, c] = np.max(a_prev_slice)
          elif mode == "average":
            A[i, h, w, c] = np.average(a_prev_slice)

    cache = (A_prev, hparameters)

    assert(A.shape == (m, n_H, n_W, n_C))

    return A, cache

In [5]:
# conv_backward

Z, cache_conv = conv_forward(A_prev, W, b, hparameters)

def conv_backward(dZ, cache):
  (A_prev, W, b, hparameters) = cache
  (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
  (f, f, n_C_prev, n_C) = W.shape

  stride = hparameters['stride']
  pad = hparameters['pad']

  (m, n_H, n_W, n_C) = dZ.shape

  dA_prev = np.zeros((m, n_H_prev, n_W_prev, n_C_prev))
  dW = np.zeros((f, f, n_C_prev, n_C))
  db = np.zeros((1, 1, 1, n_C))

  A_prev_pad = zero_pad(A_prev, pad)
  dA_prev_pad = zero_pad(dA_prev, pad)

  for i in range(m):
    a_prev_pad = A_prev_pad[i]
    da_prev_pad = dA_prev_pad[i]

    for h in range(n_H):
      for w in range(n_W):
        for c in range(n_C):

          vert_start = h * stride
          vert_end = vert_start + f
          horiz_start = w * stride
          horiz_end = horiz_start + f

          a_slice = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]

          da_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += W[:, :, :, c] * dZ[i, h, w, c]
          dW[:,:,:,c] += a_slice * dZ[i, h, w, c]
          db[:,:,:,c] += dZ[i, h, w, c]

  if pad == 0:
    dA_prev[i, :, :, :] = da_prev_pad
  else:
    dA_prev[i, :, :, :] = da_prev_pad[pad:-pad, pad:-pad]


  assert(dA_prev.shape == (m, n_H_prev, n_W_prev, n_C_prev))

  return dA_prev, dW, db


dA, dW, db = conv_backward(Z, cache_conv)

/tmp/ipykernel_29130/4007362556.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z = Z + float(b)


In [6]:
# create_mask_from_window
np.random.seed(1)
x = np.random.randn(2, 3)

def create_mask_from_window(x):
  mask = (x == np.max(x))

  return mask

mask = create_mask_from_window(x)
mask

array([[ True, False, False],
       [False, False, False]])

In [7]:
# distribute_value

def distribute_value(dz, shape):
  (n_H, n_W) = shape
  average = dz / (n_H * n_W)

  a = np.ones(shape) * average

  return a

a = distribute_value(2, (2, 2))
a

array([[0.5, 0.5],
       [0.5, 0.5]])

In [8]:
# pool_backward

def pool_backward(dA, cache, mode = "max"):
  (A_prev, hparameters) = cache

  stride = hparameters["stride"]
  f = hparameters["f"]

  m, n_H_prev, n_W_prev, n_C_prev = A_prev.shape
  m, n_H, n_W, n_C = dA.shape

  dA_prev = np.zeros((m, n_H_prev, n_W_prev, n_C_prev))

  for i in range(m):
    a_prev = A_prev[i]

    for h in range(n_H):
      for w in range(n_W):
        for c in range(n_C):
          vert_start = h * stride
          vert_end = vert_start + f
          horiz_start = w * stride
          horiz_end = horiz_start + f

          if mode == "max":
            a_prev_slice = a_prev[vert_start:vert_end, horiz_start:horiz_end, c]
            mask = (a_prev_slice == np.max(a_prev_slice))
            dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += mask * dA[i, h, w, c]

          elif mode == "average":
            da = dA[i, h, w, c]
            shape = (f, f)
            dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += distribute_value(da, shape)

  assert(dA_prev.shape == A_prev.shape)

  return dA_prev

np.random.seed(1)
A_prev = np.random.randn(5, 5, 3, 2)
hparameters = {"stride" : 1, "f": 2}
A, cache = pool_forward(A_prev, hparameters)
dA = np.random.randn(5, 4, 2, 2)

dA_prev1 = pool_backward(dA, cache, mode = "max")